# 04 — Benchmarks des modèles

**Question centrale :** Les modèles ML apportent-ils vraiment quelque chose par rapport à des baselines simples ?

## Protocole walk-forward (anti-leakage)

On utilise un protocole **walk-forward** strict, pas de cross-validation classique :

```
Train ──────────────────┤  Test (fold 1)
Train ───────────────────────┤  Test (fold 2)
Train ────────────────────────────┤  Test (fold 3)
...
Train ─────────────────────────────────────┤  Test (fold 10)
                          ↑ 2015              ↑ 2025
```

**Pourquoi ?** Pour une série temporelle, le train ne peut jamais voir le futur. La CV classique (random split) serait du leakage.

## Baselines testées

| Baseline | Prédiction |
|---|---|
| `zero_return` | Prix stable (rendement = 0) |
| `historical_mean` | Moyenne historique des rendements |
| `momentum_20d` | Dernier rendement 20j répété |
| `seasonal_naive` | Rendement moyen de ce mois sur les 5 dernières années |

## Modèles ML testés

| Modèle | Type | Features |
|---|---|---|
| `ridge_raw` | Linéaire régularisé | 249 raw features |
| `ridge_factors` | Linéaire régularisé | ~48 facteurs |
| `elasticnet_factors` | Linéaire L1+L2 | ~48 facteurs |
| `rf_factors` | Ensemble d'arbres | ~48 facteurs |
| `hgb_factors` | Gradient boosting | ~48 facteurs |
| `lgbm_factors` | Gradient boosting LightGBM | ~48 facteurs |
| `xgb_factors` | Gradient boosting XGBoost | ~48 facteurs |

In [ ]:
import sys
sys.path.insert(0, '../../src')
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
from pathlib import Path

ROOT = Path('../../')
plt.style.use('seaborn-v0_8-whitegrid')

bm = pd.read_parquet(ROOT / 'artefacts/professional_study/model_benchmarks.parquet')
print('Colonnes :', list(bm.columns))
print('Horizons :', sorted(bm['horizon'].unique()))
print('Modèles  :', list(bm['model'].unique()))

## 1. Tableau complet des résultats

In [ ]:
def color_best(s):
    is_best = s == s.min()
    return ['background-color: #c6efce; font-weight: bold' if v else '' for v in is_best]

for h in sorted(bm['horizon'].unique()):
    sub = bm[bm['horizon']==h].copy()
    sub = sub.sort_values('rmse')
    
    cols = ['model','rmse','mae','r2','directional_accuracy']
    cols = [c for c in cols if c in sub.columns]
    disp = sub[cols].copy()
    
    for c in ['rmse','mae','r2','directional_accuracy']:
        if c in disp.columns:
            disp[c] = disp[c].map(lambda x: f'{x:.4f}' if pd.notna(x) else '—')
    
    print(f'\n=== Horizon H={h}j ===')
    print(disp.to_string(index=False))

## 2. RMSE : ML vs. baselines

La question principale : est-ce que les modèles ML sont meilleurs que les baselines sur le RMSE ?

In [ ]:
baselines = ['baseline_zero_return','baseline_momentum_20d','baseline_historical_mean','baseline_seasonal_naive']
ml_models = [m for m in bm['model'].unique() if m not in baselines]

fig, axes = plt.subplots(2, 2, figsize=(18, 12))
for ax, h in zip(axes.flat, sorted(bm['horizon'].unique())):
    sub = bm[bm['horizon']==h].sort_values('rmse')
    
    colors = ['#d9534f' if m in baselines else '#5bc0de' for m in sub['model']]
    bars = ax.barh(sub['model'], sub['rmse'], color=colors, alpha=0.85)
    
    # Ligne RMSE meilleure baseline
    best_base = sub[sub['model'].isin(baselines)]['rmse'].min()
    ax.axvline(best_base, color='red', lw=1.5, ls='--', label=f'Meilleure baseline ({best_base:.4f})')
    
    ax.set_title(f'RMSE — H={h}j', fontsize=11)
    ax.set_xlabel('RMSE (plus bas = meilleur)')
    ax.legend(fontsize=9)
    
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#d9534f', alpha=0.85, label='Baseline'),
        Patch(facecolor='#5bc0de', alpha=0.85, label='Modèle ML'),
    ]
    ax.legend(handles=legend_elements + ax.get_legend_handles_labels()[0][:1])

plt.suptitle('RMSE par modèle et horizon — Rouge = baselines, Bleu = ML', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Directional Accuracy : le vrai signal utile

Pour un conseil agriculteur, **la direction compte plus que l'amplitude**.
Un modèle qui prédit systématiquement le bon sens (hausse/baisse) est utile même si ses niveaux sont imprécis.

> DA = 50% → idem pile ou face  
> DA = 60% → signal utile  
> DA = 70% → très fort signal

In [ ]:
if 'directional_accuracy' in bm.columns:
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))
    for ax, h in zip(axes.flat, sorted(bm['horizon'].unique())):
        sub = bm[bm['horizon']==h].sort_values('directional_accuracy', ascending=False)
        sub = sub[sub['directional_accuracy'].notna()]
        
        colors = ['#d9534f' if m in baselines else '#5bc0de' for m in sub['model']]
        ax.barh(sub['model'], sub['directional_accuracy'] * 100, color=colors, alpha=0.85)
        ax.axvline(50, color='gray', lw=1.5, ls='--', label='50% (aléatoire)')
        ax.axvline(55, color='orange', lw=1, ls=':', label='55% (signal faible)')
        ax.axvline(60, color='green', lw=1, ls=':', label='60% (signal utile)')
        ax.set_title(f'Directional Accuracy — H={h}j', fontsize=11)
        ax.set_xlabel('DA (%)')
        ax.set_xlim(0, 75)
        ax.legend(fontsize=8)
    
    plt.suptitle('Directional Accuracy — la vraie mesure du signal directionnel', fontsize=13)
    plt.tight_layout()
    plt.show()

## 4. Évolution de la performance dans le temps (par fold)

Est-ce que les modèles s'améliorent ou se dégradent sur les années récentes ?
Un modèle qui marche bien en 2015 mais mal en 2022 n'est pas fiable.

In [ ]:
# Chargement des prédictions détaillées pour analyse par fold
pred_path = ROOT / 'artefacts/professional_study/model_predictions.parquet'
if pred_path.exists():
    preds = pd.read_parquet(pred_path)
    print('Colonnes prédictions :', list(preds.columns)[:15])
    print(f'Lignes : {len(preds):,}')
    preds.head(3)
else:
    print('Fichier model_predictions.parquet non trouvé.')

## 5. Interprétation honnête des résultats

### Ce que les benchmarks nous disent vraiment

**Sur le RMSE :**
- H=5j : `rf_factors` (RMSE=0.0358) bat légèrement les baselines
- H=10j : `rf_factors` (RMSE=0.0495) bat légèrement les baselines
- H=20j : `baseline_seasonal_naive` gagne → **les modèles ML ne battent pas la saisonnalité à 20j**
- H=30j : même constat

**Pourquoi les baselines sont si fortes à long terme ?**

La saisonnalité du maïs est tellement structurée que la connaissance du mois courant suffit à faire une bonne prédiction à 20-30j. Les modèles ML doivent capturer ce pattern ET quelque chose en plus — et ils ne le font pas encore.

**Sur la Directional Accuracy :**
- `lgbm_factors` à H=20j : DA=61.3% → signal directionnel réel
- `rf_factors` à H=30j : DA=58.5% → signal modéré
- Les baselines non-directionnelles (zero_return, momentum) ont DA ~0-1% car elles prédisent 0

### Ce que ça implique

Le projet doit se repositionner :
- **Objectif 1 :** Prédire la direction (hausse/baisse) → LightGBM à H=20j est prometteur
- **Objectif 2 :** Quantifier l'incertitude → CQR (prochain carnet)
- **Objectif non prioritaire :** Battre les baselines en RMSE → trop dur sans données supplémentaires

**→ Prochain carnet :** Intervalles de confiance et CQR — comment mesurer l'incertitude de nos prédictions.